# 03 — Mean Reversion Strategy Analysis

This notebook analyses the Bollinger Band mean-reversion strategy with RSI
confirmation implemented in `strategies/mean_reversion.py`.

**Sections**
- Baseline run & performance metrics
- Equity curve & drawdown
- Signal and indicator analysis (Bollinger Bands + RSI per ticker)
- Signal statistics per ticker (entries, exits, holding periods)
- Parameter sensitivity: bb_window, bb_std, RSI threshold
- Exit rule comparison: exit at mean vs exit at band
- Strategy vs momentum head-to-head
- Automated key findings

**Background** — Bollinger Band mean reversion: Prices that deviate
significantly from their recent moving average tend to revert. Bollinger
Bands define 'significant deviation' as price crossing beyond N standard
deviations of a rolling window. The strategy goes long when price drops
below the lower band (oversold) and exits when price returns to the moving
average. An RSI confirmation filter avoids entering into genuine trending
moves disguised as mean-reversion opportunities — entry requires BOTH a
band breach AND RSI below the oversold threshold.

In [ ]:
%matplotlib inline
import sys
import warnings
import pathlib

_here = pathlib.Path(".").resolve()
_root = _here.parent if _here.name == "notebooks" else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

from backtester import DataLoader, Backtester, compute_metrics, drawdown_series
from strategies.mean_reversion import MeanReversionStrategy
from strategies.momentum import MomentumStrategy
from config import DEFAULT_TICKERS, BENCHMARK_TICKER, START_DATE, END_DATE, INITIAL_CAPITAL

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (14, 5)

print("Environment ready.")

In [ ]:
# ---------------------------------------------------------------------------
# Load price data — `prices` is the canonical wide-format DataFrame used
# throughout this notebook (index=DatetimeIndex, columns=ticker symbols).
# ---------------------------------------------------------------------------
all_tickers = list(DEFAULT_TICKERS) + [BENCHMARK_TICKER]

loader = DataLoader(tickers=all_tickers, start_date=START_DATE, end_date=END_DATE)
prices = loader.load_wide()   # <-- canonical variable name for this notebook

n_tickers = len(prices.columns)
n_days    = len(prices)
print(f"prices: {n_tickers} tickers × {n_days} trading days")
print(f"Date range: {prices.index.min().date()} to {prices.index.max().date()}")
prices.tail(3)

In [ ]:
# ---------------------------------------------------------------------------
# Single strategy run with default parameters.
# ---------------------------------------------------------------------------
strategy = MeanReversionStrategy()
print("Strategy:", strategy.get_name())

bt = Backtester(
    prices,
    config={
        "initial_capital": INITIAL_CAPITAL,
        "commission_bps":  5,
        "slippage_bps":    2,
        "position_sizing": "equal_weight",
        "benchmark":       BENCHMARK_TICKER,
        "allow_short":     False,
    },
)

result = bt.run(strategy)
m      = result.metrics
print(f"\nBacktest complete — {len(result.trades)} trades executed.")

In [ ]:
# ---------------------------------------------------------------------------
# Print a formatted metrics summary.
# ---------------------------------------------------------------------------
def _fmt(v):
    if v is None:            return "N/A"
    if isinstance(v, int):   return f"{v:,}"
    if isinstance(v, float): return f"{v:.4f}"
    return str(v)

SEP = "-" * 50
print(f"{'Metric':<40}  {'Value':>10}")
print(SEP)
groups = [
    ("Returns", ["total_return_pct", "annualized_return_pct",
                 "annualized_volatility_pct", "best_day_pct",
                 "worst_day_pct", "positive_days_pct"]),
    ("Risk-adjusted", ["sharpe_ratio", "sortino_ratio", "calmar_ratio"]),
    ("Drawdown", ["max_drawdown_pct", "max_drawdown_duration_days", "recovery_days"]),
    ("Activity", ["n_trades", "win_rate_pct", "avg_trade_duration_days",
                  "total_cost_pct", "turnover_annual_pct"]),
    ("vs Benchmark", ["alpha_pct", "beta", "information_ratio",
                      "correlation_with_benchmark",
                      "benchmark_annualized_return_pct",
                      "benchmark_sharpe_ratio"]),
]
for section, keys in groups:
    print(f"  {section}")
    for k in keys:
        if k in m:
            print(f"    {k:<38}  {_fmt(m[k]):>10}")
    print()

## Section 1 — Equity curve & drawdown

In [ ]:
# ---------------------------------------------------------------------------
# Equity curve vs SPY + drawdown panel.
# ---------------------------------------------------------------------------
bench_result = bt.run_benchmark()
bench_equity = bench_result.equity_curve

strat_norm = result.equity_curve / result.equity_curve.iloc[0] * 100
bench_norm = bench_equity        / bench_equity.iloc[0]        * 100
dd         = drawdown_series(result.equity_curve)

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True,
                          gridspec_kw={"height_ratios": [3, 1]})

# Top: normalised equity curves
axes[0].plot(strat_norm.index, strat_norm.values,
             color="steelblue", linewidth=1.8, label=strategy.get_name())
axes[0].plot(bench_norm.index, bench_norm.values,
             color="black", linewidth=1.4, linestyle="--",
             label=f"{BENCHMARK_TICKER} (buy & hold)")
axes[0].axhline(100, color="gray", linewidth=0.8, linestyle=":",
                label="Cash (base=100)")

# Shade regions where strategy outperforms / underperforms SPY
axes[0].fill_between(
    strat_norm.index,
    strat_norm.values,
    bench_norm.values,
    where=(strat_norm.values >= bench_norm.values),
    interpolate=True, alpha=0.15, color="green", label="Outperformance"
)
axes[0].fill_between(
    strat_norm.index,
    strat_norm.values,
    bench_norm.values,
    where=(strat_norm.values < bench_norm.values),
    interpolate=True, alpha=0.15, color="red", label="Underperformance"
)
axes[0].set_ylabel("Portfolio value (base = 100)")
axes[0].set_title("Equity curve vs. benchmark", fontweight="bold")
axes[0].legend(fontsize=9)

# Bottom: drawdown with fill
axes[1].fill_between(dd.index, dd.values, 0,
                     color="firebrick", alpha=0.4, label="Strategy drawdown")
axes[1].plot(dd.index, dd.values, color="firebrick", linewidth=0.8)
axes[1].set_ylabel("Drawdown (%)")
axes[1].set_title("Drawdown", fontweight="bold")
axes[1].legend(fontsize=9)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

plt.tight_layout()
plt.show()

## Section 2 — Signal and indicator analysis

In [ ]:
# ---------------------------------------------------------------------------
# Bollinger Band chart for the most active ticker.
# Pick ticker with highest signal activity (most non-zero signal days).
# Show a 2-year window with visible signals.
# ---------------------------------------------------------------------------
raw_signals = strategy.generate_signals(prices)

# Most active ticker
activity = (raw_signals != 0).sum()
ticker = activity.idxmax()
print(f"Most active ticker: {ticker} ({int(activity[ticker])} non-zero signal days)")

price_s = prices[ticker]
upper, middle, lower = strategy._compute_bands(price_s)
rsi = strategy._compute_rsi(price_s)
sig = raw_signals[ticker]

# Find a 2-year window with visible long signals
sig_nonzero = sig[sig != 0]
if len(sig_nonzero) > 0:
    mid_idx = len(sig_nonzero) // 2
    window_center = sig_nonzero.index[mid_idx]
    window_start = window_center - pd.DateOffset(years=1)
    window_end   = window_center + pd.DateOffset(years=1)
else:
    window_start = price_s.index[0]
    window_end   = price_s.index[min(504, len(price_s) - 1)]

mask = (price_s.index >= window_start) & (price_s.index <= window_end)
p  = price_s[mask]
ub = upper[mask]
mb = middle[mask]
lb = lower[mask]
rs = rsi[mask]
sg = sig[mask]

# Two-panel: top 70% price+bands, bottom 30% RSI
fig = plt.figure(figsize=(16, 10))
ax_price = plt.subplot2grid((10, 1), (0, 0), rowspan=7)
ax_rsi   = plt.subplot2grid((10, 1), (7, 0), rowspan=3, sharex=ax_price)

# Price + Bands
ax_price.plot(p.index, p.values, color="steelblue", linewidth=1.5, label="Close")
ax_price.plot(ub.index, ub.values, color="red",   linewidth=1.0, linestyle="--", label="Upper band")
ax_price.plot(mb.index, mb.values, color="gray",  linewidth=0.8, linestyle="-",  label="MA (middle)")
ax_price.plot(lb.index, lb.values, color="green", linewidth=1.0, linestyle="--", label="Lower band")
ax_price.fill_between(ub.index, ub.values, lb.values,
                      alpha=0.05, color="gray", label="Band region")

# Signal markers
long_days  = sg[sg == 1.0].index
short_days = sg[sg == -1.0].index
# Exit: transitions from non-zero to 0
sg_shifted = sg.shift(1).fillna(0.0)
exit_days  = sg[(sg == 0.0) & (sg_shifted != 0.0)].index

if len(long_days) > 0:
    ax_price.scatter(long_days, p.reindex(long_days),
                     color="green", s=30, zorder=5, label="Long entry")
if len(short_days) > 0:
    ax_price.scatter(short_days, p.reindex(short_days),
                     color="red", s=30, zorder=5, label="Short entry")
if len(exit_days) > 0:
    ax_price.scatter(exit_days, p.reindex(exit_days),
                     color="gray", s=20, marker="x", zorder=5, label="Exit")

ax_price.set_title(f"Bollinger Band signals — {ticker}", fontweight="bold")
ax_price.set_ylabel("Price")
ax_price.legend(fontsize=8, ncol=4)
ax_price.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax_price.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))

# RSI panel
ax_rsi.plot(rs.index, rs.values, color="darkorange", linewidth=1.2, label="RSI")
ax_rsi.axhline(strategy.rsi_oversold,   color="green", linewidth=0.8, linestyle="--",
               label=f"Oversold ({strategy.rsi_oversold})")
ax_rsi.axhline(strategy.rsi_overbought, color="red",   linewidth=0.8, linestyle="--",
               label=f"Overbought ({strategy.rsi_overbought})")
ax_rsi.axhline(50, color="gray", linewidth=0.6, linestyle=":")
ax_rsi.set_ylabel("RSI")
ax_rsi.set_ylim(0, 100)
ax_rsi.legend(fontsize=8)
ax_rsi.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# Signal statistics per ticker.
# ---------------------------------------------------------------------------
stats_rows = []

for col in raw_signals.columns:
    s = raw_signals[col]
    s_prev = s.shift(1).fillna(0.0)

    # Long entries: transition from <=0 to >0
    n_long_entries = int(((s > 0) & (s_prev <= 0)).sum())

    # Exits: transition from non-zero to 0
    n_exits = int(((s == 0.0) & (s_prev != 0.0)).sum())

    # Average holding days: mean run length of non-zero blocks
    in_position = (s != 0).astype(int)
    runs = in_position.groupby((in_position != in_position.shift()).cumsum())
    non_zero_runs = [len(grp) for _, grp in runs if grp.iloc[0] == 1]
    avg_holding = float(np.mean(non_zero_runs)) if non_zero_runs else 0.0

    pct_in_market = float((s != 0).mean() * 100)

    stats_rows.append({
        "ticker":           col,
        "n_long_entries":   n_long_entries,
        "n_exits":          n_exits,
        "avg_holding_days": round(avg_holding, 1),
        "pct_days_in_market": round(pct_in_market, 2),
    })

df_stats = (
    pd.DataFrame(stats_rows)
    .set_index("ticker")
    .sort_values("pct_days_in_market", ascending=False)
)

print(f"Overall mean pct_days_in_market : {df_stats['pct_days_in_market'].mean():.2f}%")
print(f"Overall mean avg_holding_days   : {df_stats['avg_holding_days'].mean():.1f} days")
df_stats.style.background_gradient(subset=["pct_days_in_market"], cmap="Blues")

## Section 3 — Parameter sensitivity

In [ ]:
# ---------------------------------------------------------------------------
# bb_window sensitivity.
# ---------------------------------------------------------------------------
bb_window_grid = [10, 15, 20, 30, 50]
bw_rows = []

for bw in bb_window_grid:
    s   = MeanReversionStrategy(bb_window=bw)
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(s)
    mtr = r.metrics
    bw_rows.append({
        "bb_window":             bw,
        "annualized_return_pct": mtr["annualized_return_pct"],
        "sharpe_ratio":          mtr["sharpe_ratio"],
        "max_drawdown_pct":      mtr["max_drawdown_pct"],
        "total_cost_pct":        mtr["total_cost_pct"],
        "turnover_annual_pct":   mtr["turnover_annual_pct"],
    })
    print(f"  bb_window={bw:>2}  Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%  "
          f"MaxDD={mtr['max_drawdown_pct']:.1f}%")

df_bw = pd.DataFrame(bw_rows).set_index("bb_window")

metrics_5 = ["annualized_return_pct", "sharpe_ratio", "max_drawdown_pct",
             "total_cost_pct", "turnover_annual_pct"]
labels_5  = ["Ann. Return (%)", "Sharpe ratio", "Max DD (%)",
             "Total cost (%)", "Turnover (%)"]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
x = df_bw.index.tolist()

for i, (met, lab) in enumerate(zip(metrics_5, labels_5)):
    axes[i].bar(x, df_bw[met], color="steelblue", width=3.0)
    axes[i].axvline(20, color="red", linewidth=1.2, linestyle="--", label="Default (20)")
    axes[i].set_title(lab, fontweight="bold")
    axes[i].set_xlabel("bb_window")
    axes[i].set_xticks(x)
    axes[i].legend(fontsize=8)

axes[5].set_visible(False)
fig.suptitle("Mean reversion — Bollinger Band window sensitivity",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# bb_std sensitivity.
# ---------------------------------------------------------------------------
bb_std_grid = [1.0, 1.5, 2.0, 2.5, 3.0]
bs_rows = []

for bs in bb_std_grid:
    s   = MeanReversionStrategy(bb_std=bs)
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(s)
    mtr = r.metrics
    bs_rows.append({
        "bb_std":                bs,
        "annualized_return_pct": mtr["annualized_return_pct"],
        "sharpe_ratio":          mtr["sharpe_ratio"],
        "max_drawdown_pct":      mtr["max_drawdown_pct"],
        "total_cost_pct":        mtr["total_cost_pct"],
        "turnover_annual_pct":   mtr["turnover_annual_pct"],
    })
    print(f"  bb_std={bs:.1f}  Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%  "
          f"MaxDD={mtr['max_drawdown_pct']:.1f}%")

df_bs = pd.DataFrame(bs_rows).set_index("bb_std")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
x = df_bs.index.tolist()

for i, (met, lab) in enumerate(zip(metrics_5, labels_5)):
    axes[i].bar(x, df_bs[met], color="steelblue", width=0.3)
    axes[i].axvline(2.0, color="red", linewidth=1.2, linestyle="--", label="Default (2.0)")
    axes[i].set_title(lab, fontweight="bold")
    axes[i].set_xlabel("bb_std")
    axes[i].set_xticks(x)
    axes[i].legend(fontsize=8)

axes[5].set_visible(False)
fig.suptitle("Mean reversion — band width sensitivity",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------------------
# RSI oversold threshold sensitivity.
# ---------------------------------------------------------------------------
rsi_grid = [25, 30, 35, 40, 45]
rsi_rows = []

for rv in rsi_grid:
    s   = MeanReversionStrategy(rsi_oversold=float(rv))
    eng = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                     "benchmark": BENCHMARK_TICKER})
    r   = eng.run(s)
    mtr = r.metrics
    rsi_rows.append({
        "rsi_oversold":          rv,
        "annualized_return_pct": mtr["annualized_return_pct"],
        "sharpe_ratio":          mtr["sharpe_ratio"],
        "max_drawdown_pct":      mtr["max_drawdown_pct"],
        "total_cost_pct":        mtr["total_cost_pct"],
        "turnover_annual_pct":   mtr["turnover_annual_pct"],
    })
    print(f"  rsi_oversold={rv}  Sharpe={mtr['sharpe_ratio']:.3f}  "
          f"Ann.Ret={mtr['annualized_return_pct']:.1f}%")

df_rsi = pd.DataFrame(rsi_rows).set_index("rsi_oversold")

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
x = df_rsi.index.tolist()

for i, (met, lab) in enumerate(zip(metrics_5, labels_5)):
    axes[i].bar(x, df_rsi[met], color="steelblue", width=2.5)
    axes[i].axvline(35, color="red", linewidth=1.2, linestyle="--", label="Default (35)")
    axes[i].set_title(lab, fontweight="bold")
    axes[i].set_xlabel("rsi_oversold")
    axes[i].set_xticks(x)
    axes[i].legend(fontsize=8)

axes[5].set_visible(False)
fig.suptitle("Mean reversion — RSI oversold threshold sensitivity",
             fontweight="bold", fontsize=13)
plt.tight_layout()
plt.show()

## Section 4 — Exit rule comparison

In [ ]:
# ---------------------------------------------------------------------------
# Exit at mean vs exit at band — equity curves and metrics side-by-side.
# ---------------------------------------------------------------------------
exit_mean = MeanReversionStrategy(exit_at_mean=True)
exit_band = MeanReversionStrategy(exit_at_mean=False)

eng_exit = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                      "benchmark": BENCHMARK_TICKER})
r_mean = eng_exit.run(exit_mean)
r_band = eng_exit.run(exit_band)

# Equity curves
fig, ax = plt.subplots(figsize=(14, 6))
for label, r, col in [
    ("Exit at mean", r_mean, "steelblue"),
    ("Exit at band", r_band, "darkorange"),
]:
    norm = r.equity_curve / r.equity_curve.iloc[0] * 100
    ax.plot(norm.index, norm.values, color=col, linewidth=1.8, label=label)

bench_norm_exit = bench_equity / bench_equity.iloc[0] * 100
ax.plot(bench_norm_exit.index, bench_norm_exit.values,
        color="black", linewidth=1.2, linestyle="--", label="SPY")
ax.set_title("Exit at mean vs exit at band", fontweight="bold")
ax.set_ylabel("Portfolio value (base = 100)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

# Side-by-side metrics table
compare_keys = [
    "annualized_return_pct", "sharpe_ratio", "sortino_ratio",
    "max_drawdown_pct", "calmar_ratio", "alpha_pct",
    "n_trades", "total_cost_pct", "turnover_annual_pct",
]
bench_m = bench_result.metrics

df_exit = pd.DataFrame({
    "Exit at mean": {k: r_mean.metrics.get(k) for k in compare_keys},
    "Exit at band": {k: r_band.metrics.get(k) for k in compare_keys},
    "SPY":          {k: bench_m.get(k)         for k in compare_keys},
})
df_exit

## Section 5 — Strategy vs momentum comparison

In [ ]:
# ---------------------------------------------------------------------------
# Mean reversion vs momentum head-to-head.
# Key insight: if correlation of daily returns is low, the two strategies
# are good candidates for combination in Phase 4.1.
# ---------------------------------------------------------------------------
mom_strategy = MomentumStrategy(lookback_months=12, skip_months=1, n_long=5)
mr_strategy  = MeanReversionStrategy()

engine_cmp = Backtester(prices, config={"initial_capital": INITIAL_CAPITAL,
                                        "benchmark": BENCHMARK_TICKER})
r_mom = engine_cmp.run(mom_strategy)
r_mr  = engine_cmp.run(mr_strategy)

# Daily returns from equity curves
ret_mom = r_mom.equity_curve.pct_change().dropna()
ret_mr  = r_mr.equity_curve.pct_change().dropna()

# Align on common dates
common_idx = ret_mom.index.intersection(ret_mr.index)
corr = float(ret_mom.reindex(common_idx).corr(ret_mr.reindex(common_idx)))
print(f"Pairwise return correlation (momentum vs mean reversion): {corr:.4f}")

# Equity curve overlay
fig, ax = plt.subplots(figsize=(14, 6))
for label, r, col in [
    (mom_strategy.get_name(), r_mom, "purple"),
    (mr_strategy.get_name(),  r_mr,  "teal"),
]:
    norm = r.equity_curve / r.equity_curve.iloc[0] * 100
    ax.plot(norm.index, norm.values, linewidth=1.8, label=label, color=col)

ax.plot(bench_norm_exit.index, bench_norm_exit.values,
        color="black", linewidth=1.2, linestyle="--", label="SPY")
ax.set_title("Momentum vs Mean Reversion vs SPY", fontweight="bold")
ax.set_ylabel("Portfolio value (base = 100)")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Combined metrics table
df_vs = pd.DataFrame({
    "Momentum":       {k: r_mom.metrics.get(k) for k in compare_keys},
    "Mean Reversion": {k: r_mr.metrics.get(k)  for k in compare_keys},
    "SPY":            {k: bench_m.get(k)        for k in compare_keys},
})
df_vs

## Section 6 — Key findings

In [ ]:
# ---------------------------------------------------------------------------
# Automated findings — derived programmatically from the runs above.
# ---------------------------------------------------------------------------
base_mtr  = result.metrics
bench_mtr = bench_result.metrics

best_bw_idx = df_bw["sharpe_ratio"].idxmax()
best_bs_idx = df_bs["sharpe_ratio"].idxmax()
best_rv_idx = df_rsi["sharpe_ratio"].idxmax()

exit_mean_sharpe = float(r_mean.metrics["sharpe_ratio"])
exit_band_sharpe = float(r_band.metrics["sharpe_ratio"])
best_exit = "exit_at_mean" if exit_mean_sharpe >= exit_band_sharpe else "exit_at_band"

divers_direction = "increase" if corr < 0.3 else "decrease"

SEP = "=" * 60
print(SEP)
print("MEAN REVERSION STRATEGY FINDINGS")
print(SEP)
print(f"  Default spec (bb=20/2.0σ, rsi=14/35/65, exit=mean, binary)")
print(f"    Annualized return  : {base_mtr['annualized_return_pct']:.2f}%")
print(f"    Sharpe ratio       : {base_mtr['sharpe_ratio']:.4f}")
print(f"    Max drawdown       : {base_mtr['max_drawdown_pct']:.2f}%")
if "alpha_pct" in base_mtr:
    print(f"    Alpha vs SPY       : {base_mtr['alpha_pct']:.2f}%")
print()
print(f"  Best bb_window       : {best_bw_idx}  "
      f"(Sharpe = {df_bw.loc[best_bw_idx, 'sharpe_ratio']:.4f})")
print(f"  Best bb_std          : {best_bs_idx}  "
      f"(Sharpe = {df_bs.loc[best_bs_idx, 'sharpe_ratio']:.4f})")
print(f"  Best rsi_oversold    : {best_rv_idx}  "
      f"(Sharpe = {df_rsi.loc[best_rv_idx, 'sharpe_ratio']:.4f})")
print()
print(f"  Exit rule comparison :")
print(f"    exit_at_mean Sharpe: {exit_mean_sharpe:.4f}")
print(f"    exit_at_band Sharpe: {exit_band_sharpe:.4f}")
print(f"    → {best_exit} achieves higher Sharpe")
print()
print(f"  Correlation with momentum strategy: {corr:.4f}")
print(f"  Combining with momentum would {divers_direction} diversification "
      f"based on {corr:.2f} pairwise return correlation.")

## Notes

_Fill in manual observations after running the notebook:_

- Default strategy performance vs SPY:
- Best Bollinger Band window from sensitivity:
- Best band width (bb_std) from sensitivity:
- Best RSI oversold threshold from sensitivity:
- Exit rule winner and rationale:
- Correlation with momentum strategy and diversification implication: